# 2.a Benchmark — easy-search + 评估

对 `work/DB/` 中各方法库跑检索（query = target），写出灵敏度表与 `auc_easy.csv`（**绘图在 `3.plot.ipynb`**）。

- Foldseek / 预测方法：`bin/foldseek easy-search`，`-s 9.5 --max-seqs 2000 -e 10`；灵敏度用 **hitlist** 协议（TSV 内分母，对齐 `new_scope40`）
- MMseqs2：`bin/mmseqs search` + `convertalis`，参数对齐 `foldseek-analysis`：`-s 7.5 --max-seqs 2000 -e 10000 -a`；灵敏度用 **catalog** 协议（`bench.noselfhit.awk`：分母为库内全部同源，且只平均有效 query）

共享配置：[`config.py`](config.py) 的 `EASY_SEARCH_PARAMS` / `MMSEQS_SEARCH_PARAMS`。

> 检索较吃 CPU。登录节点可把 `THREADS` 调小。MMseqs 参数变更后需重搜：`MMSEQS_SKIP_EXISTING = False`。


In [1]:
from __future__ import annotations

import gc
import os
import re
import subprocess
import traceback
from pathlib import Path

import pandas as pd

from collections import defaultdict, Counter

from config import (
    AA_FASTA,
    ALN_DIR,
    DBS_DIR,
    EASY_SEARCH_PARAMS,
    FOLDSEEK_BIN,
    METRICS_DIR,
    METHODS,
    MMSEQS_BIN,
    MMSEQS_SEARCH_PARAMS,
    PROJECT_ROOT,
    SCOP_LOOKUP,
    WORK_DIR,
    aln_tmp_dir,
    aln_tsv,
    db_prefix,
    cleanup_tmp,
    ensure_work_dirs,
    method_engine,
    method_eval_protocol,
    metric_prefix,
    require_project_root,
    scop_cla_path,
)

ROOT = require_project_root("2.a.benchmark.ipynb")
assert ROOT == PROJECT_ROOT

SKIP_EXISTING = True
# 已改为 foldseek-analysis 的 -s 7.5 -e 10000；旧 mmseqs_easy.tsv 需重搜
MMSEQS_SKIP_EXISTING = False
THREADS = EASY_SEARCH_PARAMS["threads"]  # 可改为 8 / 16
ONLY_METHODS = None  # 例如 ["foldseek", "mmseqs", "ESM3_LoRA"]；None = 全部

ensure_work_dirs()
print("ROOT:", ROOT)
print("foldseek:", FOLDSEEK_BIN)
print("mmseqs:", MMSEQS_BIN)
print("DB:", DBS_DIR)
print("方法:", [(k, eng, method_eval_protocol(k)) for _, k, eng, _ in METHODS])
print("THREADS =", THREADS)
print("MMseqs params:", MMSEQS_SEARCH_PARAMS)
print("MMSEQS_SKIP_EXISTING =", MMSEQS_SKIP_EXISTING)


ROOT: /hpcfs/fhome/caihuize/scope40_easy
foldseek: /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek
mmseqs: /hpcfs/fhome/caihuize/scope40_easy/bin/mmseqs
DB: /hpcfs/fhome/caihuize/scope40_easy/work/DB
方法: [('foldseek', 'foldseek'), ('mmseqs', 'mmseqs'), ('ESM3', 'foldseek'), ('ESM3_LoRA', 'foldseek'), ('ProstT5', 'foldseek'), ('SaProt', 'foldseek')]
THREADS = 64


## Phase A — 检索

输出：`work/aln/{method}_easy.tsv`

- `foldseek` / 预测方法 → Foldseek easy-search（`EASY_SEARCH_PARAMS`）
- `mmseqs` → MMseqs2 search + convertalis（`MMSEQS_SEARCH_PARAMS`，对齐 foldseek-analysis）


In [2]:
def foldseek_easy_search(
    query_db: Path,
    output_tsv: Path,
    tmp_dir: Path,
    target_db: Path | None = None,
    threads: int | None = None,
    skip_existing: bool = True,
) -> Path:
    target_db = Path(target_db or query_db)
    threads = int(threads if threads is not None else EASY_SEARCH_PARAMS["threads"])

    if skip_existing and output_tsv.is_file() and output_tsv.stat().st_size > 0:
        print(f"⏭️  比对已存在，跳过: {output_tsv} ({output_tsv.stat().st_size} bytes)")
        return output_tsv

    if not FOLDSEEK_BIN.is_file():
        raise FileNotFoundError(f"foldseek 不存在: {FOLDSEEK_BIN}")
    if not query_db.is_file():
        raise FileNotFoundError(f"query DB 不存在: {query_db}")
    if not target_db.is_file():
        raise FileNotFoundError(f"target DB 不存在: {target_db}")

    output_tsv.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(FOLDSEEK_BIN), "easy-search",
        str(query_db), str(target_db), str(output_tsv), str(tmp_dir),
        "--threads", str(threads),
        "-s", str(EASY_SEARCH_PARAMS["sensitivity"]),
        "--max-seqs", str(EASY_SEARCH_PARAMS["max_seqs"]),
        "-e", str(EASY_SEARCH_PARAMS["evalue"]),
    ]
    print("[CMD]", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if not output_tsv.is_file():
        raise FileNotFoundError(f"结果未生成: {output_tsv}")
    print(f"✅ {output_tsv}")
    return output_tsv


def mmseqs_easy_search(
    query_db: Path,
    output_tsv: Path,
    tmp_dir: Path,
    target_db: Path | None = None,
    threads: int | None = None,
    skip_existing: bool = True,
) -> Path:
    """Search an existing MMseqs DB. `easy-search` only accepts FASTA, so use search + convertalis.

    Parameters match foldseek-analysis/scopbenchmark/scripts/runMMseqs.sh:
    `-a --threads N -s 7.5 -e 10000 --max-seqs 2000`.
    """
    import shutil

    target_db = Path(target_db or query_db)
    threads = int(threads if threads is not None else MMSEQS_SEARCH_PARAMS["threads"])
    params = MMSEQS_SEARCH_PARAMS

    if skip_existing and output_tsv.is_file() and output_tsv.stat().st_size > 0:
        print(f"⏭️  比对已存在，跳过: {output_tsv} ({output_tsv.stat().st_size} bytes)")
        return output_tsv

    if not MMSEQS_BIN.is_file():
        raise FileNotFoundError(
            f"mmseqs 不存在: {MMSEQS_BIN}\n请先运行 0.prepare.ipynb 下载 MMseqs"
        )
    if not query_db.is_file():
        raise FileNotFoundError(f"query DB 不存在: {query_db}")
    if not target_db.is_file():
        raise FileNotFoundError(f"target DB 不存在: {target_db}")

    output_tsv.parent.mkdir(parents=True, exist_ok=True)
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    search_tmp = tmp_dir / "search_tmp"
    result_db = tmp_dir / "result"
    search_tmp.mkdir(parents=True, exist_ok=True)

    cmd_search = [
        str(MMSEQS_BIN), "search",
        str(query_db), str(target_db), str(result_db), str(search_tmp),
        "--threads", str(threads),
        "-s", str(params["sensitivity"]),
        "--max-seqs", str(params["max_seqs"]),
        "-e", str(params["evalue"]),
    ]
    if params.get("add_backtrace"):
        cmd_search.append("-a")
    print("[CMD]", " ".join(cmd_search))
    subprocess.run(cmd_search, check=True)

    cmd_conv = [
        str(MMSEQS_BIN), "convertalis",
        str(query_db), str(target_db), str(result_db), str(output_tsv),
        "--threads", str(threads),
        "--format-output", "query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits",
    ]
    print("[CMD]", " ".join(cmd_conv))
    subprocess.run(cmd_conv, check=True)

    if not output_tsv.is_file() or output_tsv.stat().st_size == 0:
        raise FileNotFoundError(f"结果未生成: {output_tsv}")
    print(f"✅ {output_tsv}")
    return output_tsv


def search_one(method_key: str, skip_existing: bool = True, threads: int | None = None) -> Path:
    ensure_work_dirs()
    engine = method_engine(method_key)
    if engine == "mmseqs":
        skip_existing = MMSEQS_SKIP_EXISTING
        if threads is None:
            threads = int(MMSEQS_SEARCH_PARAMS["threads"])
    kwargs = dict(
        query_db=db_prefix(method_key),
        output_tsv=aln_tsv(method_key),
        tmp_dir=aln_tmp_dir(method_key),
        skip_existing=skip_existing,
        threads=threads,
    )
    if engine == "mmseqs":
        return mmseqs_easy_search(**kwargs)
    if engine == "foldseek":
        return foldseek_easy_search(**kwargs)
    raise ValueError(f"未知 engine: {engine} ({method_key})")


def search_all(skip_existing: bool = True, threads: int | None = None) -> dict[str, Path]:
    out: dict[str, Path] = {}
    for _label, key, engine, _di in METHODS:
        print(f"\n══ easy-search: {key} ({engine}) ══")
        out[key] = search_one(key, skip_existing=skip_existing, threads=threads)
    return out


if ONLY_METHODS is None:
    aln_paths = search_all(skip_existing=SKIP_EXISTING, threads=THREADS)
else:
    aln_paths = {}
    for key in ONLY_METHODS:
        print(f"\n══ easy-search: {key} ({method_engine(key)}) ══")
        aln_paths[key] = search_one(key, skip_existing=SKIP_EXISTING, threads=THREADS)

for key, path in aln_paths.items():
    nlines = sum(1 for _ in path.open()) if path.is_file() else 0
    print(f"{key:12s} → {path.name}  lines={nlines:,}")



══ easy-search: foldseek (foldseek) ══
⏭️  比对已存在，跳过: /hpcfs/fhome/caihuize/scope40_easy/work/aln/foldseek_easy.tsv (446853701 bytes)

══ easy-search: mmseqs (mmseqs) ══
[CMD] /hpcfs/fhome/caihuize/scope40_easy/bin/mmseqs search /hpcfs/fhome/caihuize/scope40_easy/work/DB/mmseqs_DB/DB /hpcfs/fhome/caihuize/scope40_easy/work/DB/mmseqs_DB/DB /hpcfs/fhome/caihuize/scope40_easy/work/tmp/easy_mmseqs/result /hpcfs/fhome/caihuize/scope40_easy/work/tmp/easy_mmseqs/search_tmp --threads 64 -s 9.5 --max-seqs 2000 -e 10.0
search /hpcfs/fhome/caihuize/scope40_easy/work/DB/mmseqs_DB/DB /hpcfs/fhome/caihuize/scope40_easy/work/DB/mmseqs_DB/DB /hpcfs/fhome/caihuize/scope40_easy/work/tmp/easy_mmseqs/result /hpcfs/fhome/caihuize/scope40_easy/work/tmp/easy_mmseqs/search_tmp --threads 64 -s 9.5 --max-seqs 2000 -e 10.0 

MMseqs Version:                        	8cc5ce367b5638c4306c2d7cfc652dd099a4643f
Substitution matrix                    	aa:blosum62.out,nucl:nucleotide.out
Add backtrace                    

## Phase B — 评估

1. 使用 `work/lable/scop_lookup.tsv`（由 `0.prepare` 生成；缺失时可重建）
2. 写出 `*_fam/sup/fol.tsv`
3. 汇总 `work/metrics/auc_easy.csv`

灵敏度协议（`config.method_eval_protocol`）：

- Foldseek / 预测 3Di：`hitlist`（比对 TSV 内的同类 hit 为分母）
- MMseqs2：`catalog`（foldseek-analysis `bench.noselfhit.awk`：库内全部同源为分母；只保留同时有 family / 远程 sfam / 远程 fold 成员的 query；零命中记 0）


In [3]:
def remove_family_number(scop_class: str) -> str:
    return re.sub(r"\.[0-9]+$", "", scop_class)


def resolve_scop_class(qid: str, id2cls: dict[str, str]) -> str | None:
    candidates: list[str] = [qid]
    base_model = re.sub(r"_MODEL_.*", "", qid)
    if base_model != qid:
        candidates.append(base_model)
    for cand in list(candidates):
        if "." in cand:
            base_chain = re.sub(r"_[A-Za-z0-9]+$", "", cand)
            if base_chain != cand:
                candidates.append(base_chain)
    for cand in candidates:
        if cand in id2cls:
            return id2cls[cand]
    return None


def build_scop_lookup(skip_existing: bool = True) -> Path:
    scop_cla = scop_cla_path()
    ensure_work_dirs()
    if skip_existing and SCOP_LOOKUP.is_file() and SCOP_LOOKUP.stat().st_size > 0:
        print(f"⏭️  SCOP lookup 已存在: {SCOP_LOOKUP}")
        return SCOP_LOOKUP

    id2cls: dict[str, str] = {}
    with scop_cla.open() as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split("\t")
            if len(parts) >= 4:
                id2cls[parts[0].strip()] = parts[3].strip()
    print(f"从 dir.cla 读取了 {len(id2cls)} 个 domain")

    all_ids: set[str] = set()
    with AA_FASTA.open() as f:
        for line in f:
            if line.startswith(">"):
                qid = line[1:].strip().split()[0]
                if qid:
                    all_ids.add(qid)
    print(f"FASTA 中共有 {len(all_ids)} 个唯一 ID")

    missed = 0
    with SCOP_LOOKUP.open("w") as out:
        for qid in sorted(all_ids):
            cls = resolve_scop_class(qid, id2cls)
            if cls is None:
                missed += 1
                continue
            out.write(f"{qid}\t{cls}\n")
    print(f"✅ {SCOP_LOOKUP}  匹配={len(all_ids) - missed} 未匹配={missed}")
    return SCOP_LOOKUP


def load_scop_levels() -> pd.DataFrame:
    rows = []
    with SCOP_LOOKUP.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            fam = parts[1].strip()
            sf = remove_family_number(fam)
            fo = remove_family_number(sf)
            rows.append({"id": parts[0].strip(), "fa": fam, "sf": sf, "fo": fo})
    return pd.DataFrame(rows)


def calc_fp_rates(aln_tsv_path: Path, cla: pd.DataFrame, out_prefix: Path) -> dict[str, Path]:
    print(f"  读取比对: {aln_tsv_path}", flush=True)
    aln = pd.read_csv(aln_tsv_path, sep="\t", header=None, usecols=[0, 1], names=["qid", "tid"], dtype=str)
    print(f"  原始行数: {len(aln):,}", flush=True)
    aln = aln[aln["qid"] != aln["tid"]].copy()
    print(f"  去 self-hit 后: {len(aln):,}", flush=True)

    work = aln.merge(cla, left_on="qid", right_on="id", how="inner")
    work = work.rename(columns={"fo": "qfo", "sf": "qsf", "fa": "qfa"}).drop(columns=["id"])
    work = work.merge(cla, left_on="tid", right_on="id", how="left")
    work = work.rename(columns={"fo": "tfo", "sf": "tsf", "fa": "tfa"}).drop(columns=["id"])

    is_wrong_fold = (work["qfo"] != work["tfo"]).fillna(True)
    work["seen_fp"] = is_wrong_fold.groupby(work["qid"], sort=False).cumsum().gt(0).astype("int8")

    same_fo = work["qfo"] == work["tfo"]
    same_sf = work["qsf"] == work["tsf"]
    same_fa = work["qfa"] == work["tfa"]

    count_fold = (same_fo & ~same_sf).astype("int32")
    count_super = (same_fo & same_sf & ~same_fa).astype("int32")
    count_family = (same_fo & same_sf & same_fa).astype("int32")

    before_fp = 1 - work["seen_fp"]
    work["fold_tp"] = count_fold * before_fp
    work["super_tp"] = count_super * before_fp
    work["family_tp"] = count_family * before_fp
    work["count_fold"] = count_fold
    work["count_super"] = count_super
    work["count_family"] = count_family

    agg = (
        work.groupby("qid", sort=False)
        .agg(
            focnt=("fold_tp", "sum"), fotot=("count_fold", "sum"),
            sfcnt=("super_tp", "sum"), sftot=("count_super", "sum"),
            facnt=("family_tp", "sum"), fatot=("count_family", "sum"),
        )
        .reset_index()
    )
    for tot in ("fotot", "sftot", "fatot"):
        agg[tot] = agg[tot].replace(0, 1)
    agg["fofrac"] = agg["focnt"] / agg["fotot"]
    agg["sfrac"] = agg["sfcnt"] / agg["sftot"]
    agg["fafrac"] = agg["facnt"] / agg["fatot"]

    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    paths = {
        "fol": Path(str(out_prefix) + "_fol.tsv"),
        "sup": Path(str(out_prefix) + "_sup.tsv"),
        "fam": Path(str(out_prefix) + "_fam.tsv"),
    }
    agg[["qid", "focnt", "fotot", "fofrac"]].to_csv(paths["fol"], sep="\t", header=False, index=False)
    agg[["qid", "sfcnt", "sftot", "sfrac"]].to_csv(paths["sup"], sep="\t", header=False, index=False)
    agg[["qid", "facnt", "fatot", "fafrac"]].to_csv(paths["fam"], sep="\t", header=False, index=False)
    return paths


def calc_fp_rates_catalog(aln_tsv_path: Path, cla: pd.DataFrame, out_prefix: Path) -> dict[str, Path]:
    """foldseek-analysis bench.noselfhit.awk：分母 = 库内该层级全部同源。

    - 跳过 lookup 中没有的 query
    - target 不在 lookup → FP（可多次累加，与 awk 一致）
    - 跳过 self-hit
    - 第一个 wrong-fold 之后不再计 TP
    - Family / Superfamily / Fold 互斥计数
    - 只输出同时有 family、远程 sfam、远程 fold 成员的 query
    - 分母与 awk 相同：famCnt-1、sfamCnt-(famCnt-1)、foldCnt-(sfamCnt-1)
    """
    print(f"  catalog 协议（bench.noselfhit.awk）: {aln_tsv_path}", flush=True)
    id2fam = dict(zip(cla["id"].astype(str), cla["fa"].astype(str)))
    id2sfam = dict(zip(cla["id"].astype(str), cla["sf"].astype(str)))
    id2fold = dict(zip(cla["id"].astype(str), cla["fo"].astype(str)))
    fam_cnt: dict[str, int] = Counter(cla["fa"].astype(str))
    sfam_cnt: dict[str, int] = Counter(cla["sf"].astype(str))
    fold_cnt: dict[str, int] = Counter(cla["fo"].astype(str))

    found_fam: dict[str, int] = defaultdict(int)
    found_sfam: dict[str, int] = defaultdict(int)
    found_fold: dict[str, int] = defaultdict(int)
    found_fp: dict[str, int] = defaultdict(int)

    n_lines = 0
    with aln_tsv_path.open() as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            qid, tid = parts[0], parts[1]
            n_lines += 1
            if qid not in id2fam:
                continue
            if tid not in id2fam:
                found_fp[qid] += 1
                continue
            if qid == tid:
                continue
            q_fam, q_sfam, q_fold = id2fam[qid], id2sfam[qid], id2fold[qid]
            t_fam, t_sfam, t_fold = id2fam[tid], id2sfam[tid], id2fold[tid]
            if found_fp[qid] < 1 and q_fold != t_fold:
                found_fp[qid] += 1
                continue
            if found_fp[qid] < 1 and q_fam == t_fam:
                found_fam[qid] += 1
                continue
            if found_fp[qid] < 1 and q_fam != t_fam and q_sfam == t_sfam:
                found_sfam[qid] += 1
                continue
            if (
                found_fp[qid] < 1
                and q_fam != t_fam
                and q_sfam != t_sfam
                and q_fold == t_fold
            ):
                found_fold[qid] += 1
                continue

    print(f"  原始行数: {n_lines:,}", flush=True)

    rows_fam: list[tuple[str, int, int, float]] = []
    rows_sup: list[tuple[str, int, int, float]] = []
    rows_fol: list[tuple[str, int, int, float]] = []
    n_valid = 0
    for qid, fam in id2fam.items():
        if not fam:
            continue
        sfam = id2sfam[qid]
        fold = id2fold[qid]
        n_fam = fam_cnt[fam]
        n_sfam = sfam_cnt[sfam]
        n_fold = fold_cnt[fold]
        if not (n_fam > 1 and n_sfam - n_fam > 0 and n_fold - n_sfam > 0):
            continue
        n_valid += 1
        fam_tot = n_fam - 1
        sfam_tot = n_sfam - (n_fam - 1)
        fold_tot = n_fold - (n_sfam - 1)
        fam_tp = found_fam[qid]
        sfam_tp = found_sfam[qid]
        fold_tp = found_fold[qid]
        rows_fam.append((qid, fam_tp, fam_tot, fam_tp / fam_tot))
        rows_sup.append((qid, sfam_tp, sfam_tot, sfam_tp / sfam_tot))
        rows_fol.append((qid, fold_tp, fold_tot, fold_tp / fold_tot))

    print(f"  有效 query: {n_valid:,} / lookup {len(id2fam):,}", flush=True)
    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    paths = {
        "fol": Path(str(out_prefix) + "_fol.tsv"),
        "sup": Path(str(out_prefix) + "_sup.tsv"),
        "fam": Path(str(out_prefix) + "_fam.tsv"),
    }

    def _write(path: Path, rows: list[tuple[str, int, int, float]]) -> None:
        with path.open("w") as out:
            for qid, cnt, tot, frac in rows:
                out.write(f"{qid}\t{cnt}\t{tot}\t{frac:.6f}\n")

    _write(paths["fam"], rows_fam)
    _write(paths["sup"], rows_sup)
    _write(paths["fol"], rows_fol)
    return paths


def mean_sensitivity(level_tsv: Path) -> float:
    vals = []
    with level_tsv.open() as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                vals.append(float(parts[3]))
    return sum(vals) / len(vals) if vals else 0.0


def evaluate_all(skip_existing: bool = True) -> pd.DataFrame:
    ensure_work_dirs()
    build_scop_lookup(skip_existing=skip_existing)
    cla = load_scop_levels()
    print(f"SCOP levels: {len(cla)}")

    auc_rows: list[dict] = []
    for level_name, level_key in (("Family", "fam"), ("Superfamily", "sup"), ("Fold", "fol")):
        row: dict[str, float | str] = {"search_mode": "easy", "level": level_name}
        for label, key, _engine, _di in METHODS:
            tsv_path = aln_tsv(key)
            out_prefix = metric_prefix(key)
            fam_path = Path(str(out_prefix) + "_fam.tsv")
            level_path = Path(str(out_prefix) + f"_{level_key}.tsv")

            print(f"\n[easy] {label}  protocol={method_eval_protocol(key)}")
            if not tsv_path.is_file():
                print(f"  ❌ 缺少比对: {tsv_path}")
                continue

            protocol = method_eval_protocol(key)
            # catalog 结果若仍是旧 hitlist 表会错，MMseqs 始终重算（评估很快）
            skip_this = (
                protocol != "catalog"
                and skip_existing
                and fam_path.is_file()
                and fam_path.stat().st_size > 0
            )
            if not skip_this:
                if level_name == "Family":
                    try:
                        if protocol == "catalog":
                            calc_fp_rates_catalog(tsv_path, cla, out_prefix)
                            print("  ✅ catalog 写入 fam/sup/fol")
                        else:
                            calc_fp_rates(tsv_path, cla, out_prefix)
                            print("  ✅ hitlist 写入 fam/sup/fol")
                    except Exception as e:
                        print(f"  ❌ {e}")
                        traceback.print_exc()
                        continue
                    finally:
                        gc.collect()
            else:
                if level_name == "Family":
                    print("  ⏭️  评估已存在")

            if level_path.is_file():
                row[label] = mean_sensitivity(level_path)
                print(f"  {level_name} AUC={row[label]:.4f}")
        auc_rows.append(row)

    df = pd.DataFrame(auc_rows)
    csv_path = METRICS_DIR / "auc_easy.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n✅ AUC CSV: {csv_path}")
    return df


auc_df = evaluate_all(skip_existing=SKIP_EXISTING)
display(auc_df)
print("\nBenchmark 完成。下一步打开 3.plot.ipynb（作图）")


⏭️  SCOP lookup 已存在: /hpcfs/fhome/caihuize/scope40_easy/work/lable/scop_lookup.tsv
SCOP levels: 13920

[easy] Foldseek (AA+3Di)
  读取比对: /hpcfs/fhome/caihuize/scope40_easy/work/aln/foldseek_easy.tsv
  原始行数: 7,847,383
  去 self-hit 后: 7,833,466
  ✅ 写入 fam/sup/fol
  Family AUC=0.7354

[easy] MMseqs2
  读取比对: /hpcfs/fhome/caihuize/scope40_easy/work/aln/mmseqs_easy.tsv
  原始行数: 296,366
  去 self-hit 后: 282,446
  ✅ 写入 fam/sup/fol
  Family AUC=0.6256

[easy] ESM3-3Di
  读取比对: /hpcfs/fhome/caihuize/scope40_easy/work/aln/ESM3_easy.tsv
  原始行数: 11,834,886
  去 self-hit 后: 11,820,968
  ✅ 写入 fam/sup/fol
  Family AUC=0.6946

[easy] ESM3-LoRA
  读取比对: /hpcfs/fhome/caihuize/scope40_easy/work/aln/ESM3_LoRA_easy.tsv
  原始行数: 10,177,070
  去 self-hit 后: 10,163,153
  ✅ 写入 fam/sup/fol
  Family AUC=0.6985

[easy] ProstT5 (translate)
  读取比对: /hpcfs/fhome/caihuize/scope40_easy/work/aln/ProstT5_easy.tsv
  原始行数: 9,427,751
  去 self-hit 后: 9,413,834
  ✅ 写入 fam/sup/fol
  Family AUC=0.7077

[easy] SaProt
  读取比对: /hpcfs/fhom

,search_mode,level,Foldseek (AA+3Di),MMseqs2,ESM3-3Di,ESM3-LoRA,ProstT5 (translate),SaProt
0,easy,Family,0.735449,0.625569,0.694610,0.698518,0.707741,0.457892
1,easy,Superfamily,0.631220,0.527703,0.582254,0.581927,0.589313,0.343547
2,easy,Fold,0.080475,0.012281,0.054260,0.051478,0.055377,0.024268



Benchmark 完成。下一步打开 3.plot.ipynb（作图）


## 清理临时目录

删除项目根 `tmp/` 与 `work/tmp/`（下载/解压/搜索中间文件）。产物在 `work/` 与 `bin/` 中保留。


In [4]:
cleanup_tmp(also_work_tmp=True)


🧹 已清理: /hpcfs/fhome/caihuize/scope40_easy/tmp
🧹 已清理: /hpcfs/fhome/caihuize/scope40_easy/work/tmp
